# Machine Learning Model Training

This notebook walks through the ML pipeline implemented in `scripts/ml/`:
- Feature engineering (polynomial, datetime, binning, target encoding)
- Training multiple classifiers with cross-validation
- Hyper-parameter tuning with GridSearchCV
- Evaluation: metrics, confusion matrix, ROC curve, feature importances

In [ ]:
# !pip install -r ../requirements.txt

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

from scripts.ml.feature_engineering import add_polynomial_features, bin_column, log_transform
from scripts.ml.model_training import train_with_cross_validation, tune_hyperparameters, save_model, MODEL_REGISTRY
from scripts.ml.model_evaluation import classification_metrics, print_classification_report, plot_confusion_matrix, plot_feature_importances

%matplotlib inline

## 1. Dataset

In [ ]:
X_arr, y_arr = make_classification(
    n_samples=1000, n_features=15, n_informative=8,
    n_classes=2, random_state=42
)
X = pd.DataFrame(X_arr, columns=[f'feature_{i}' for i in range(X_arr.shape[1])])
y = pd.Series(y_arr, name='target')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 2. Feature Engineering

In [ ]:
# Add polynomial interaction features for the first 3 features
X_train_fe = add_polynomial_features(X_train, columns=['feature_0', 'feature_1', 'feature_2'], degree=2)
X_test_fe  = add_polynomial_features(X_test,  columns=['feature_0', 'feature_1', 'feature_2'], degree=2)
print('Shape after feature engineering:', X_train_fe.shape)

## 3. Model Training with Cross-Validation

In [ ]:
results = {}
for model_name in MODEL_REGISTRY:
    pipeline, cv_scores = train_with_cross_validation(
        X_train_fe, y_train, model_name=model_name, cv=5
    )
    results[model_name] = {'pipeline': pipeline, 'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std()}
    print(f'{model_name}: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

## 4. Evaluation on Test Set

In [ ]:
best_name = max(results, key=lambda k: results[k]['cv_mean'])
best_pipeline = results[best_name]['pipeline']
print(f'Best model: {best_name}')

y_pred = best_pipeline.predict(X_test_fe)
y_prob = best_pipeline.predict_proba(X_test_fe)[:, 1]

metrics = classification_metrics(y_test.values, y_pred, y_prob)
print(metrics)
print_classification_report(y_test.values, y_pred)

In [ ]:
plot_confusion_matrix(y_test.values, y_pred)

In [ ]:
plot_feature_importances(best_pipeline, feature_names=X_train_fe.columns.tolist(), top_n=15)

## 5. Hyper-parameter Tuning

In [ ]:
grid_result = tune_hyperparameters(X_train_fe, y_train, model_name='random_forest', cv=3)
print('Best params:', grid_result.best_params_)
print('Best CV score:', grid_result.best_score_)

y_pred_tuned = grid_result.best_estimator_.predict(X_test_fe)
print(classification_metrics(y_test.values, y_pred_tuned))